# Experiment 5: Geometry Exploration for Eikonal Bent-Ray

**Goal**: Understand the transducer geometry and L-matrix structure before implementing the eikonal bent-ray L-matrix update.

**What we need to know:**
1. Physical coordinate grids (SoS 64x64 and DT 128x128) — ranges, spacing, units (meters)
2. Transducer element positions — 128 elements, 300μm pitch (Rau et al. 2021)
3. Firing pair configuration — 8 pairs, δ_channel = 17 element separation
4. L-matrix structure per pair — how rows map to beamforming pixels, sensitivity patterns
5. Verify: can we reconstruct the straight-ray geometry from L alone?

**Reference**: Rau, R. et al. (2021) "Speed-of-sound imaging using diverging waves"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.sparse import csc_matrix
import h5py

from inr_sos import DATA_DIR
from inr_sos.io.utils import load_mat, load_L_matrix, load_sample, load_metadata

In [ ]:
# File paths
DATA_FILE = DATA_DIR + "/DL-based-SoS/test_kWaveGeom_l2rec_l1rec_unifiedvar.mat"
GRID_FILE = DATA_DIR + "/DL-based-SoS/forward_model_lr/grid_parameters.mat"

# Load grid parameters
grid = load_mat(GRID_FILE)
print("Grid keys:", [k for k in grid.keys() if not k.startswith('__')])

## 1. Physical Coordinate Grids

The grid_parameters.mat contains:
- `xax_sos`, `zax_sos`: 1D axes for the 64x64 SoS reconstruction grid (meters)
- `xDT`, `zDT`: 1D axes for the 128x128 DT (displacement/beamforming) grid (meters)
- `xdt`, `zdt`: 2D meshgrids (128x128) for the DT grid

In [ ]:
# Extract 1D axes
x_sos = grid['xax_sos'].flatten()  # (64,) SoS grid lateral positions
z_sos = grid['zax_sos'].flatten()  # (64,) SoS grid depths
x_dt = grid['xDT'].flatten()       # (128,) DT grid lateral positions
z_dt = grid['zDT'].flatten()       # (128,) DT grid depths

# Extract 2D meshgrids
xdt_2d = grid['xdt']  # (128, 128)
zdt_2d = grid['zdt']  # (128, 128)

print("=== SoS Reconstruction Grid (64x64) ===")
print(f"  x_sos: [{x_sos[0]:.6f}, {x_sos[-1]:.6f}] m  |  {len(x_sos)} points  |  spacing: {np.diff(x_sos).mean()*1e3:.4f} mm")
print(f"  z_sos: [{z_sos[0]:.6f}, {z_sos[-1]:.6f}] m  |  {len(z_sos)} points  |  spacing: {np.diff(z_sos).mean()*1e3:.4f} mm")
print(f"  Field of view: {(x_sos[-1]-x_sos[0])*1e3:.2f} x {(z_sos[-1]-z_sos[0])*1e3:.2f} mm")

print(f"\n=== DT Beamforming Grid (128x128) ===")
print(f"  x_dt: [{x_dt[0]:.6f}, {x_dt[-1]:.6f}] m  |  {len(x_dt)} points  |  spacing: {np.diff(x_dt).mean()*1e3:.4f} mm")
print(f"  z_dt: [{z_dt[0]:.6f}, {z_dt[-1]:.6f}] m  |  {len(z_dt)} points  |  spacing: {np.diff(z_dt).mean()*1e3:.4f} mm")
print(f"  Field of view: {(x_dt[-1]-x_dt[0])*1e3:.2f} x {(z_dt[-1]-z_dt[0])*1e3:.2f} mm")

print(f"\n=== 2D DT Meshgrids ===")
print(f"  xdt_2d shape: {xdt_2d.shape}, zdt_2d shape: {zdt_2d.shape}")
print(f"  xdt_2d range: [{xdt_2d.min():.6f}, {xdt_2d.max():.6f}] m")
print(f"  zdt_2d range: [{zdt_2d.min():.6f}, {zdt_2d.max():.6f}] m")

In [ ]:
# Visualize the grids
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) SoS grid points
ax = axes[0]
X_sos, Z_sos = np.meshgrid(x_sos, z_sos)
ax.scatter(X_sos.flatten()*1e3, Z_sos.flatten()*1e3, s=0.5, alpha=0.5)
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title(f"SoS Grid (64x64)\nspacing: {np.diff(x_sos).mean()*1e3:.3f} mm")
ax.set_aspect('equal')
ax.invert_yaxis()

# (b) DT grid points (subsample for visibility)
ax = axes[1]
ax.scatter(xdt_2d.flatten()*1e3, zdt_2d.flatten()*1e3, s=0.2, alpha=0.3)
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title(f"DT Grid (128x128)\nspacing: {np.diff(x_dt).mean()*1e3:.3f} mm")
ax.set_aspect('equal')
ax.invert_yaxis()

# (c) Overlay both grids
ax = axes[2]
ax.scatter(X_sos.flatten()*1e3, Z_sos.flatten()*1e3, s=1, alpha=0.3, label='SoS 64x64', color='blue')
ax.scatter(xdt_2d.flatten()*1e3, zdt_2d.flatten()*1e3, s=0.2, alpha=0.1, label='DT 128x128', color='red')
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title("Overlay: SoS + DT grids")
ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend()

plt.tight_layout()
plt.show()

## 2. Transducer Element Positions

From Rau et al. (2021):
- 128-element linear array, 300 μm pitch
- Array is at the surface (z ≈ 0), centered laterally
- 8 firing pairs with element separation δ_channel = 17

The element lateral positions should be: `x_elem[n] = (n - 63.5) × 0.3e-3` for n = 0..127, centered at x=0.

In [ ]:
# Transducer geometry from Rau et al. (2021)
N_ELEM = 128
PITCH = 0.3e-3  # 300 μm in meters
DELTA_CH = 17   # element separation for firing pairs

# Element positions (centered array)
elem_x = (np.arange(N_ELEM) - (N_ELEM - 1) / 2) * PITCH  # centered at x=0
elem_z = 0.0  # at surface

print(f"=== Transducer Array ===")
print(f"  {N_ELEM} elements, pitch = {PITCH*1e3:.1f} mm")
print(f"  Array extent: [{elem_x[0]*1e3:.2f}, {elem_x[-1]*1e3:.2f}] mm")
print(f"  Array center: {elem_x.mean()*1e3:.4f} mm")
print(f"  Total aperture: {(elem_x[-1] - elem_x[0])*1e3:.2f} mm")

# Firing pairs: 8 pairs evenly spaced across the array
# From the paper: pairs separated by δ_channel = 17 elements (~5.1mm)
# Typical arrangement: pairs centered across the array
# The exact pair indices aren't in the paper, but we can infer from 8 pairs x 128 elements
# Standard: elements are grouped symmetrically. Let's compute likely pairs.

# If pairs are evenly spaced:
# With 128 elements and 8 pairs, the center elements would be at indices:
# pair_center[p] = (p + 0.5) * 128/8 = (p + 0.5) * 16 for p in 0..7
# elem_i = pair_center - delta_ch/2, elem_j = pair_center + delta_ch/2

print(f"\n=== Firing Pairs (δ_channel = {DELTA_CH}) ===")
print(f"  Element separation: {DELTA_CH * PITCH * 1e3:.2f} mm")

# We'll confirm the exact pair indices from the L-matrix analysis below.
# For now, assume evenly spaced centers:
for p in range(8):
    center = (p + 0.5) * (N_ELEM / 8)
    ei = int(center - DELTA_CH / 2)
    ej = int(center + DELTA_CH / 2)
    print(f"  Pair {p}: elem {ei} (x={elem_x[ei]*1e3:.2f}mm) -- elem {ej} (x={elem_x[ej]*1e3:.2f}mm)  |  Δx = {(elem_x[ej]-elem_x[ei])*1e3:.2f}mm")

In [ ]:
# Visualize: transducer elements relative to imaging grids
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# SoS grid
ax.scatter(X_sos.flatten()*1e3, Z_sos.flatten()*1e3, s=0.3, alpha=0.15, color='blue', label='SoS 64x64')

# Transducer elements
ax.scatter(elem_x*1e3, np.zeros(N_ELEM), s=15, color='red', zorder=5, label='Elements')

# Highlight firing pair elements (assumed positions)
for p in range(8):
    center = (p + 0.5) * (N_ELEM / 8)
    ei = int(center - DELTA_CH / 2)
    ej = int(center + DELTA_CH / 2)
    color = plt.cm.tab10(p)
    ax.plot([elem_x[ei]*1e3, elem_x[ej]*1e3], [0, 0], '-o', color=color, 
            markersize=8, linewidth=2, label=f'Pair {p}' if p < 4 else None)

ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title("Transducer Array + SoS Grid Geometry")
ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(elem_x[0]*1e3 - 2, elem_x[-1]*1e3 + 2)
plt.tight_layout()
plt.show()

## 3. L-Matrix Structure Analysis

The L-matrix has shape (131072, 4096) = (8 × 16384, 64 × 64).
- 8 firing pairs, each producing 128×128 = 16384 displacement measurements
- Each row encodes the differential path length: `L_row = path(elem_i → pixel) - path(elem_j → pixel)`
- Row ordering: MATLAB column-major, so pixel (ix, iz) maps to row `iz * 128 + ix` within each pair block

Let's analyze each pair's L-matrix block to understand the sensitivity patterns.

In [ ]:
# Load L-matrix and metadata
L = load_L_matrix(DATA_FILE)
meta = load_metadata(DATA_FILE)

print(f"L-matrix shape: {L.shape}")
print(f"L-matrix dtype: {L.dtype}")
print(f"L-matrix range: [{L.min():.6e}, {L.max():.6e}]")
print(f"L-matrix nnz fraction: {(L != 0).sum() / L.size:.4f}")
print(f"\nMetadata: bf_sos = {meta.get('bf_sos')}, pix2time = {meta.get('pix2time')}")

N_PAIRS = 8
ROWS_PER_PAIR = 128 * 128  # 16384
N_PIXELS = 64 * 64  # 4096

assert L.shape == (N_PAIRS * ROWS_PER_PAIR, N_PIXELS), f"Unexpected L shape: {L.shape}"

In [ ]:
# Per-pair statistics
print(f"{'Pair':>4} | {'nnz%':>8} | {'min':>12} | {'max':>12} | {'mean':>12} | {'std':>12} | {'row_norm_mean':>14}")
print("-" * 90)
for p in range(N_PAIRS):
    L_pair = L[p * ROWS_PER_PAIR : (p + 1) * ROWS_PER_PAIR, :]
    nnz_frac = (L_pair != 0).sum() / L_pair.size
    row_norms = np.linalg.norm(L_pair, axis=1)
    print(f"{p:>4} | {nnz_frac:>7.4f}% | {L_pair.min():>12.4e} | {L_pair.max():>12.4e} | "
          f"{L_pair.mean():>12.4e} | {L_pair.std():>12.4e} | {row_norms.mean():>14.4e}")

## 4. L-Matrix Sensitivity Maps per Pair

For each firing pair, sum |L| across all DT pixels (rows) to see which SoS pixels are most sensitive.
This reveals the "footprint" of each pair on the reconstruction grid.

Also: sum |L| across SoS pixels (columns) to see the sensitivity of each DT beamforming pixel.

In [ ]:
# Column-sum sensitivity: for each pair, which SoS pixels are most affected?
# sum(|L_pair|, axis=0) → (4096,) → reshape to (64, 64) with order='F' (MATLAB convention)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for p in range(N_PAIRS):
    ax = axes[p // 4, p % 4]
    L_pair = L[p * ROWS_PER_PAIR : (p + 1) * ROWS_PER_PAIR, :]
    col_sens = np.abs(L_pair).sum(axis=0)  # (4096,)
    img = col_sens.reshape(64, 64, order='F')
    im = ax.imshow(img, aspect='equal', cmap='hot',
                   extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
    ax.set_title(f"Pair {p}", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    if p % 4 == 0:
        ax.set_ylabel("z (mm)")
    if p >= 4:
        ax.set_xlabel("x (mm)")

fig.suptitle("Column Sensitivity: Σ|L| over DT rows → SoS grid (64×64)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Row-sum sensitivity: for each pair, which DT pixels have the most influence?
# sum(|L_pair|, axis=1) → (16384,) → reshape to (128, 128) with order='F'
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for p in range(N_PAIRS):
    ax = axes[p // 4, p % 4]
    L_pair = L[p * ROWS_PER_PAIR : (p + 1) * ROWS_PER_PAIR, :]
    row_sens = np.abs(L_pair).sum(axis=1)  # (16384,)
    img = row_sens.reshape(128, 128, order='F')
    im = ax.imshow(img, aspect='equal', cmap='hot',
                   extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
    ax.set_title(f"Pair {p}", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    if p % 4 == 0:
        ax.set_ylabel("z (mm)")
    if p >= 4:
        ax.set_xlabel("x (mm)")

fig.suptitle("Row Sensitivity: Σ|L| over SoS cols → DT grid (128×128)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Single-Row L Patterns (Individual DT Pixels)

Visualize what a single row of L looks like when reshaped to the 64×64 SoS grid.
This shows the "ray path" — which SoS pixels contribute to a given DT displacement measurement.

Pick a few representative DT pixel locations: center, corner, edge.

In [ ]:
# Show individual L-matrix rows for pair 0 at selected DT pixel locations
# DT pixel (ix, iz) → row index within pair: iz * 128 + ix (column-major/Fortran order)

pair_idx = 0
L_pair = L[pair_idx * ROWS_PER_PAIR : (pair_idx + 1) * ROWS_PER_PAIR, :]

# Selected DT pixel positions (ix, iz) — 0-indexed
dt_pixels = [
    (64, 64, "center"),
    (32, 32, "upper-left"),
    (96, 96, "lower-right"),
    (64, 32, "top-center"),
    (64, 96, "bottom-center"),
    (32, 64, "mid-left"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (ix, iz, label) in enumerate(dt_pixels):
    ax = axes[i // 3, i % 3]
    row_idx = iz * 128 + ix  # Fortran order within pair block
    L_row = L_pair[row_idx, :]
    img = L_row.reshape(64, 64, order='F')
    
    vmax = max(abs(img.min()), abs(img.max()))
    if vmax == 0:
        vmax = 1e-10
    
    im = ax.imshow(img, aspect='equal', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
    
    # Mark the DT pixel position on the SoS grid
    dt_x_pos = x_dt[ix] * 1e3
    dt_z_pos = z_dt[iz] * 1e3
    ax.plot(dt_x_pos, dt_z_pos, 'g+', markersize=15, markeredgewidth=2)
    
    ax.set_title(f"Pair {pair_idx}, DT({ix},{iz}) — {label}\nΣ|L|={np.abs(L_row).sum():.4e}, nnz={np.count_nonzero(L_row)}")
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Single L-matrix rows → SoS sensitivity (64×64)\nGreen + = DT pixel location", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Infer Element Positions from L-Matrix

The L-matrix encodes differential paths: `d_row = path(elem_i → scatterer) - path(elem_j → scatterer)`.

For a straight-ray model, each path is the Euclidean distance. The sign pattern of L reveals which element is "closer" to each SoS pixel. The **zero-crossing line** (where L ≈ 0) should be the perpendicular bisector of the two elements — giving us the element positions.

Strategy: For each pair, find the lateral position where the column sensitivity switches sign → this is the midpoint between the two elements.

In [ ]:
# Signed column-sum: average L value per SoS pixel (not abs), reveals sign structure
# Positive = elem_i farther, Negative = elem_j farther
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pair_centers = []

for p in range(N_PAIRS):
    ax = axes[p // 4, p % 4]
    L_pair = L[p * ROWS_PER_PAIR : (p + 1) * ROWS_PER_PAIR, :]
    col_mean = L_pair.mean(axis=0)  # signed average
    img = col_mean.reshape(64, 64, order='F')
    
    vmax = max(abs(img.min()), abs(img.max()))
    im = ax.imshow(img, aspect='equal', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
    
    # Find zero-crossing in lateral direction (average over depth)
    lateral_profile = img.mean(axis=0)  # average over z
    # Find where sign changes
    sign_changes = np.where(np.diff(np.sign(lateral_profile)))[0]
    if len(sign_changes) > 0:
        # Interpolate zero crossing
        idx = sign_changes[len(sign_changes)//2]  # take middle crossing
        x0 = x_sos[idx] + (x_sos[idx+1] - x_sos[idx]) * (-lateral_profile[idx]) / (lateral_profile[idx+1] - lateral_profile[idx])
        pair_centers.append(x0)
        ax.axvline(x0*1e3, color='lime', linewidth=1, linestyle='--')
    else:
        pair_centers.append(np.nan)
    
    ax.set_title(f"Pair {p}", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Signed Column Mean of L per Pair → Sign Structure\nDashed green = estimated pair center", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Print inferred pair centers
print("\n=== Inferred Pair Centers (from L-matrix zero crossings) ===")
for p, xc in enumerate(pair_centers):
    print(f"  Pair {p}: center x = {xc*1e3:.2f} mm")

In [ ]:
# From the pair centers, estimate element positions
# Each pair center = midpoint of (elem_i, elem_j) with separation DELTA_CH * PITCH
half_sep = DELTA_CH * PITCH / 2

print("=== Estimated Element Positions per Pair ===")
print(f"  Element separation: {DELTA_CH * PITCH * 1e3:.2f} mm (half: {half_sep*1e3:.2f} mm)")
print()
for p, xc in enumerate(pair_centers):
    if not np.isnan(xc):
        x_left = xc - half_sep
        x_right = xc + half_sep
        # Find closest element indices
        idx_left = np.argmin(np.abs(elem_x - x_left))
        idx_right = np.argmin(np.abs(elem_x - x_right))
        print(f"  Pair {p}: center={xc*1e3:>7.2f} mm  →  elem_i={idx_left:>3d} (x={elem_x[idx_left]*1e3:>7.2f}mm)  "
              f"elem_j={idx_right:>3d} (x={elem_x[idx_right]*1e3:>7.2f}mm)  Δelem={idx_right-idx_left}")
    else:
        print(f"  Pair {p}: no zero crossing found")

## 7. Mask Analysis per Pair

The mask tells us which measurement rows are valid (~54% valid overall). Let's see how the mask distributes per pair — this reveals the physical coverage of each firing pair.

In [ ]:
# Load a sample to get the mask
sample = load_sample(DATA_FILE, idx=0)
mask = sample['mask']
print(f"Mask shape: {mask.shape}, valid fraction: {mask.mean():.4f}")

# Per-pair mask visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for p in range(N_PAIRS):
    ax = axes[p // 4, p % 4]
    mask_pair = mask[p * ROWS_PER_PAIR : (p + 1) * ROWS_PER_PAIR]
    img = mask_pair.reshape(128, 128, order='F')
    valid_frac = mask_pair.mean()
    
    ax.imshow(img, aspect='equal', cmap='gray',
              extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
    ax.set_title(f"Pair {p} ({valid_frac:.1%} valid)", fontsize=10)
    if p % 4 == 0:
        ax.set_ylabel("z (mm)")
    if p >= 4:
        ax.set_xlabel("x (mm)")

fig.suptitle("Validity Mask per Pair (white = valid, black = invalid)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Straight-Ray L Reconstruction

To verify our understanding, let's manually construct L for one pair using straight-ray geometry and compare against the actual L-matrix.

For a differential measurement at beamforming pixel (x_p, z_p):
- `path_i = distance(elem_i, pixel) / c_bg` (one-way travel time through each SoS pixel)
- `path_j = distance(elem_j, pixel) / c_bg`
- `L_row = ∂(path_i - path_j)/∂s` for each SoS pixel

For a straight ray, the path length through a SoS pixel is the chord length of the ray through that pixel.

In [ ]:
# Simple straight-ray L verification for pair 0
# For diverging waves, each element fires → wave hits scatterer → reflects → received
# The ONE-WAY path from element to beamforming pixel position is what matters
# L_row[j] = path_length_through_SoS_pixel_j(elem_i→pixel) - path_length_through_SoS_pixel_j(elem_j→pixel)
#
# In the simplest model: L_row is the DIFFERENCE in Euclidean distance from 
# each element to the scatterer, discretized onto the SoS grid.
# But the actual L includes sub-pixel rasterization (anti-aliased path weighting).
#
# Let's compute the simple distance difference and compare the SIGN PATTERN with the real L.

pair_idx = 0
L_pair = L[pair_idx * ROWS_PER_PAIR : (pair_idx + 1) * ROWS_PER_PAIR, :]

# Estimated element positions for pair 0
xc = pair_centers[0]
x_ei = xc - half_sep  # left element
x_ej = xc + half_sep  # right element
z_elem = 0.0  # elements at surface

print(f"Pair {pair_idx}: elem_i at x={x_ei*1e3:.2f}mm, elem_j at x={x_ej*1e3:.2f}mm")

# For each DT pixel, compute distance to each element
X_dt_2d, Z_dt_2d = np.meshgrid(x_dt, z_dt)  # (128, 128)
dist_i = np.sqrt((X_dt_2d - x_ei)**2 + (Z_dt_2d - z_elem)**2)  # (128, 128)
dist_j = np.sqrt((X_dt_2d - x_ej)**2 + (Z_dt_2d - z_elem)**2)  # (128, 128)
diff_dist = dist_i - dist_j  # positive where elem_i is farther

# The REAL L-matrix has this distance discretized onto the SoS grid.
# But the sign pattern should match: where the real L average is positive/negative
# should correspond to where elem_i is farther/closer.

# Compare: reshape real L column mean vs distance difference
real_sign = L_pair.mean(axis=0).reshape(64, 64, order='F')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
im = ax.imshow(real_sign, aspect='equal', cmap='RdBu_r',
               extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
ax.set_title("Real L: signed column mean")
plt.colorbar(im, ax=ax)

ax = axes[1]
im = ax.imshow(diff_dist, aspect='equal', cmap='RdBu_r',
               extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
ax.set_title("dist(elem_i) - dist(elem_j)\non DT grid")
plt.colorbar(im, ax=ax)

# Resample distance difference to SoS grid (simple nearest)
from scipy.interpolate import RegularGridInterpolator
interp = RegularGridInterpolator((z_dt, x_dt), diff_dist, method='linear', bounds_error=False, fill_value=0)
Z_sos_2d, X_sos_2d = np.meshgrid(z_sos, x_sos, indexing='ij')
diff_on_sos = interp((Z_sos_2d, X_sos_2d))

ax = axes[2]
im = ax.imshow(diff_on_sos, aspect='equal', cmap='RdBu_r',
               extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
ax.set_title("dist diff resampled to SoS grid")
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# Check sign correlation
sign_corr = np.corrcoef(np.sign(real_sign.flatten()), np.sign(diff_on_sos.flatten()))[0, 1]
print(f"\nSign correlation (real L vs straight-ray distance diff): {sign_corr:.4f}")

## 9. L-Matrix Row Structure: Understanding the Rasterization

Each L-matrix row should look like a "ray path" traced from the element to the beamforming pixel, discretized on the 64×64 SoS grid. Let's look at individual rows more carefully to understand how the path is rasterized (Bresenham, anti-aliased, etc.).

In [ ]:
# Zoom into single L rows for pair 0 at a specific DT pixel
# This reveals the ray-path rasterization pattern

pair_idx = 0
L_pair = L[pair_idx * ROWS_PER_PAIR : (pair_idx + 1) * ROWS_PER_PAIR, :]

# Pick a DT pixel near center
ix_dt, iz_dt = 64, 64  # center pixel
row_idx = iz_dt * 128 + ix_dt
L_row = L_pair[row_idx, :]
img = L_row.reshape(64, 64, order='F')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Full view
ax = axes[0]
vmax = max(abs(img.min()), abs(img.max()))
im = ax.imshow(img, aspect='equal', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
ax.plot(x_dt[ix_dt]*1e3, z_dt[iz_dt]*1e3, 'g+', markersize=15, markeredgewidth=2, label='DT pixel')
# Mark estimated element positions
xc = pair_centers[0]
ax.plot((xc - half_sep)*1e3, 0, 'rv', markersize=10, label='elem_i')
ax.plot((xc + half_sep)*1e3, 0, 'b^', markersize=10, label='elem_j')
ax.set_title(f"L row for pair {pair_idx}, DT({ix_dt},{iz_dt})")
ax.legend(fontsize=8)
plt.colorbar(im, ax=ax)

# Positive part only (elem_i path contribution)
ax = axes[1]
pos_img = np.maximum(img, 0)
im = ax.imshow(pos_img, aspect='equal', cmap='Reds',
               extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
ax.plot(x_dt[ix_dt]*1e3, z_dt[iz_dt]*1e3, 'g+', markersize=15, markeredgewidth=2)
ax.plot((xc - half_sep)*1e3, 0, 'rv', markersize=10)
ax.set_title("Positive part (elem_i path longer)")
plt.colorbar(im, ax=ax)

# Negative part only (elem_j path contribution)
ax = axes[2]
neg_img = np.maximum(-img, 0)
im = ax.imshow(neg_img, aspect='equal', cmap='Blues',
               extent=[x_sos[0]*1e3, x_sos[-1]*1e3, z_sos[-1]*1e3, z_sos[0]*1e3])
ax.plot(x_dt[ix_dt]*1e3, z_dt[iz_dt]*1e3, 'g+', markersize=15, markeredgewidth=2)
ax.plot((xc + half_sep)*1e3, 0, 'b^', markersize=10)
ax.set_title("Negative part (elem_j path longer)")
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# Statistics
print(f"Row nnz: {np.count_nonzero(L_row)}/{len(L_row)}")
print(f"Positive entries: {(L_row > 0).sum()}, Negative: {(L_row < 0).sum()}")
print(f"Sum: {L_row.sum():.6e} (should be ~ diff in total path length)")
print(f"|L_row| range: [{np.abs(L_row[L_row!=0]).min():.4e}, {np.abs(L_row).max():.4e}]")

## 10. Grid Alignment Check

Critical for eikonal: verify that the SoS and DT grids are spatially aligned.
- Do the SoS grid boundaries contain the DT grid, or vice versa?
- Where is z=0 (transducer surface) relative to the grids?
- Is the element array centered on the lateral extent of both grids?

In [ ]:
print("=== Grid Alignment Summary ===")
print(f"\nTransducer array:")
print(f"  x: [{elem_x[0]*1e3:.3f}, {elem_x[-1]*1e3:.3f}] mm  (aperture: {(elem_x[-1]-elem_x[0])*1e3:.2f} mm)")
print(f"  z: {elem_z*1e3:.3f} mm (surface)")

print(f"\nSoS grid (64×64):")
print(f"  x: [{x_sos[0]*1e3:.3f}, {x_sos[-1]*1e3:.3f}] mm  (width: {(x_sos[-1]-x_sos[0])*1e3:.2f} mm)")
print(f"  z: [{z_sos[0]*1e3:.3f}, {z_sos[-1]*1e3:.3f}] mm  (depth: {(z_sos[-1]-z_sos[0])*1e3:.2f} mm)")
print(f"  Pixel size: {np.diff(x_sos).mean()*1e3:.4f} × {np.diff(z_sos).mean()*1e3:.4f} mm")

print(f"\nDT grid (128×128):")
print(f"  x: [{x_dt[0]*1e3:.3f}, {x_dt[-1]*1e3:.3f}] mm  (width: {(x_dt[-1]-x_dt[0])*1e3:.2f} mm)")
print(f"  z: [{z_dt[0]*1e3:.3f}, {z_dt[-1]*1e3:.3f}] mm  (depth: {(z_dt[-1]-z_dt[0])*1e3:.2f} mm)")
print(f"  Pixel size: {np.diff(x_dt).mean()*1e3:.4f} × {np.diff(z_dt).mean()*1e3:.4f} mm")

print(f"\n=== Key Observations ===")
print(f"  SoS grid starts at z = {z_sos[0]*1e3:.3f} mm (gap from surface: {z_sos[0]*1e3:.3f} mm)")
print(f"  DT grid starts at z = {z_dt[0]*1e3:.3f} mm")
print(f"  SoS x centered at: {(x_sos[0]+x_sos[-1])/2*1e3:.4f} mm")
print(f"  DT x centered at: {(x_dt[0]+x_dt[-1])/2*1e3:.4f} mm")
print(f"  Array x centered at: {elem_x.mean()*1e3:.4f} mm")

# Check if grids are co-located
print(f"\n  SoS grid contained in DT grid laterally? {x_dt[0] <= x_sos[0] and x_sos[-1] <= x_dt[-1]}")
print(f"  SoS grid contained in DT grid in depth? {z_dt[0] <= z_sos[0] and z_sos[-1] <= z_dt[-1]}")

## 11. Measurement Domain: Ground Truth Forward Pass

Verify the forward model: compute `d_pred = L @ s_gt` and compare against `d_meas`.
This shows the mismatch that motivates the eikonal bent-ray approach.

In [ ]:
# Load sample 0 and compute forward pass
s_gt = sample['s_gt']  # (4096,) ground truth slowness
d_meas = sample['d_meas']  # (131072,) measured displacement

# Forward pass with straight-ray L
d_pred = L @ s_gt  # (131072,)

# Mismatch
residual = d_meas - d_pred
mask_flat = mask

# Masked statistics
valid = mask_flat > 0.5
print(f"=== Forward Model Mismatch (sample 0) ===")
print(f"  d_meas range: [{d_meas[valid].min():.6e}, {d_meas[valid].max():.6e}]")
print(f"  d_pred range: [{d_pred[valid].min():.6e}, {d_pred[valid].max():.6e}]")
print(f"  residual (valid): mean={residual[valid].mean():.4e}, std={residual[valid].std():.4e}")
print(f"  relative error: {np.abs(residual[valid]).mean() / np.abs(d_meas[valid]).mean():.4%}")

# Visualize mismatch per pair
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for p in range(N_PAIRS):
    sl = slice(p * ROWS_PER_PAIR, (p + 1) * ROWS_PER_PAIR)
    m_pair = mask_flat[sl]
    d_m = d_meas[sl]
    d_p = d_pred[sl]
    r = residual[sl]
    
    # Measured
    ax = axes[0, p]
    img = d_m.reshape(128, 128, order='F')
    im = ax.imshow(img, aspect='equal', cmap='RdBu_r',
                   extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
    ax.set_title(f"P{p} meas", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    
    # Predicted
    ax = axes[1, p]
    img = d_p.reshape(128, 128, order='F')
    im = ax.imshow(img, aspect='equal', cmap='RdBu_r',
                   extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
    ax.set_title(f"P{p} pred", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    
    # Residual
    ax = axes[2, p]
    img = r.reshape(128, 128, order='F')
    vmax_r = np.abs(r[m_pair > 0.5]).max() if (m_pair > 0.5).any() else 1
    im = ax.imshow(img, aspect='equal', cmap='RdBu_r', vmin=-vmax_r, vmax=vmax_r,
                   extent=[x_dt[0]*1e3, x_dt[-1]*1e3, z_dt[-1]*1e3, z_dt[0]*1e3])
    ax.set_title(f"P{p} resid", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

axes[0, 0].set_ylabel("Measured")
axes[1, 0].set_ylabel("Predicted")
axes[2, 0].set_ylabel("Residual")
fig.suptitle("Forward Model: d_meas vs L@s_gt per Pair", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 12. Summary & Next Steps

Collect all the geometry parameters we've learned to inform the eikonal bent-ray implementation.

In [ ]:
print("=" * 70)
print("  GEOMETRY SUMMARY FOR EIKONAL BENT-RAY IMPLEMENTATION")
print("=" * 70)

print(f"""
Transducer:
  - {N_ELEM} elements, pitch = {PITCH*1e6:.0f} μm ({PITCH*1e3:.1f} mm)
  - Array at z = 0 (surface)
  - Array extent: [{elem_x[0]*1e3:.2f}, {elem_x[-1]*1e3:.2f}] mm

Firing Pairs:
  - 8 pairs, δ_channel = {DELTA_CH} elements ({DELTA_CH*PITCH*1e3:.2f} mm)
  - Pair centers (inferred from L): see above

SoS Reconstruction Grid (64×64):
  - x: [{x_sos[0]*1e3:.3f}, {x_sos[-1]*1e3:.3f}] mm, spacing: {np.diff(x_sos).mean()*1e3:.4f} mm
  - z: [{z_sos[0]*1e3:.3f}, {z_sos[-1]*1e3:.3f}] mm, spacing: {np.diff(z_sos).mean()*1e3:.4f} mm

DT Beamforming Grid (128×128):
  - x: [{x_dt[0]*1e3:.3f}, {x_dt[-1]*1e3:.3f}] mm, spacing: {np.diff(x_dt).mean()*1e3:.4f} mm
  - z: [{z_dt[0]*1e3:.3f}, {z_dt[-1]*1e3:.3f}] mm, spacing: {np.diff(z_dt).mean()*1e3:.4f} mm

Physical Constants:
  - bf_sos = {meta.get('bf_sos', 'N/A')} m/s
  - pix2time = {meta.get('pix2time', 'N/A'):.6e} s

L-matrix:
  - Shape: {L.shape}
  - 8 blocks of {ROWS_PER_PAIR} rows (128×128 DT pixels per pair)
  - Row ordering: Fortran (column-major), pixel (ix, iz) → row iz*128 + ix
  - Each row: differential path length discretized on 64×64 SoS grid

For Eikonal Implementation:
  1. Solve eikonal |∇T|² = s² on the 64×64 SoS grid per source element
  2. Source positions: element locations (x_elem, z=0)
  3. Backtrace from each DT beamforming pixel along -∇T
  4. Compute path lengths through SoS pixels → build L_bent
  5. L_bent_row = path_i - path_j for each DT pixel
""")

# Save key geometry params for use in scripts
geometry = {
    'n_elem': N_ELEM,
    'pitch': PITCH,
    'delta_ch': DELTA_CH,
    'elem_x': elem_x.tolist(),
    'x_sos': x_sos.tolist(),
    'z_sos': z_sos.tolist(),
    'x_dt': x_dt.tolist(),
    'z_dt': z_dt.tolist(),
    'bf_sos': meta.get('bf_sos'),
    'pix2time': meta.get('pix2time'),
    'pair_centers_inferred': [float(x) if not np.isnan(x) else None for x in pair_centers],
}
print("Geometry dict keys:", list(geometry.keys()))